# Day 5 — Pipeline Automation & Data Quality

This notebook demonstrates the end-to-end Heat-Pulse pipeline in:
1. **Full mode** — re-ingest everything, clean, engineer features.
2. **Incremental mode** — detect that data is already up-to-date and skip gracefully.
3. **Architecture diagram** — a text-based flowchart of the pipeline.


## 0  Imports & path setup

In [ ]:
import sys, os
from pathlib import Path

# Make sure the src/ folder is importable
PROJECT_ROOT = Path.cwd().parent   # adjust if you run from a different folder
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

# Paths used throughout
DB_PATH   = PROJECT_ROOT / 'data' / 'weather.duckdb'
DATA_DIR  = PROJECT_ROOT / 'data'
LOG_DIR   = PROJECT_ROOT / 'logs'

print('Project root:', PROJECT_ROOT)
print('DB path     :', DB_PATH)


## 1  Full pipeline run

Equivalent to:
```bash
python src/pipeline.py --mode full
```


In [ ]:
from pipeline import run_pipeline

summary_full = run_pipeline(
    mode       = 'full',
    db_path    = DB_PATH,
    data_dir   = DATA_DIR,
    log_dir    = LOG_DIR,
    start_date = '2020-01-01',
)

print('\n── Full run summary ──────────────────────')
for k, v in summary_full.items():
    print(f'  {k:<20}: {v}')


## 2  Inspect database after full run

In [ ]:
import duckdb
import pandas as pd

conn = duckdb.connect(str(DB_PATH))

from database import print_row_counts
print_row_counts(conn)

# Quick peek at analytics layer
try:
    df_analytics = conn.execute('SELECT * FROM analytics_historical LIMIT 5').df()
    print('analytics_historical columns:', list(df_analytics.columns))
    display(df_analytics)
except Exception as e:
    print(f'analytics_historical not available: {e}')

conn.close()


## 3  Incremental pipeline run

Equivalent to:
```bash
python src/pipeline.py --mode incremental
```

Because we just ran a full load, the pipeline should detect that all cities are
already up-to-date and skip the API fetch gracefully.


In [ ]:
summary_inc = run_pipeline(
    mode     = 'incremental',
    db_path  = DB_PATH,
    data_dir = DATA_DIR,
    log_dir  = LOG_DIR,
)

print('\n── Incremental run summary ───────────────')
for k, v in summary_inc.items():
    print(f'  {k:<20}: {v}')

print(f'\nCities skipped (already up-to-date): {summary_inc.get("cities_skipped", 0)}')
print(f'New rows ingested: {summary_inc.get("rows_ingested", 0)}')


## 4  Quality gate results (standalone demo)

In [ ]:
import duckdb
from quality_checks import run_all_checks, print_quality_report

conn = duckdb.connect(str(DB_PATH))

try:
    raw_df      = conn.execute('SELECT * FROM raw_historical').df()
except Exception:
    raw_df = None

try:
    staging_df  = conn.execute('SELECT * FROM staging_historical').df()
except Exception:
    staging_df = None

try:
    analytics_df = conn.execute('SELECT * FROM analytics_historical').df()
except Exception:
    analytics_df = None

conn.close()

results = run_all_checks(
    raw_df       = raw_df,
    staging_df   = staging_df,
    analytics_df = analytics_df,
)
print_quality_report(results)

# Display as a DataFrame for notebook clarity
pd.DataFrame(results)


## 5  View pipeline log

In [ ]:
log_file = LOG_DIR / 'pipeline.log'

if log_file.exists():
    lines = log_file.read_text(encoding='utf-8').splitlines()
    # Show last 40 lines
    print(f'... (showing last 40 of {len(lines)} lines) ...\n')
    print('\n'.join(lines[-40:]))
else:
    print(f'Log file not found at {log_file}')


## 6  Pipeline Architecture Diagram

```
╔══════════════════════════════════════════════════════════════════════════╗
║             HEAT-PULSE  —  End-to-End Pipeline Architecture              ║
╚══════════════════════════════════════════════════════════════════════════╝

  ┌─────────────────────────────────────────────────────────────────────┐
  │                        TRIGGER (CLI)                                │
  │   python src/pipeline.py --mode [full | incremental]                │
  └───────────────────────────────┬─────────────────────────────────────┘
                                  │
                                  ▼
  ┌─────────────────────────────────────────────────────────────────────┐
  │  STAGE 1 — INGEST            (src/ingestion.py + pipeline.py)       │
  │                                                                     │
  │  full mode       → fetch ALL dates from Open-Meteo API              │
  │  incremental     → check max(time) per city in raw_historical        │
  │                    fetch only dates AFTER that                      │
  │                    APPEND (not replace) new rows                    │
  │                                                                     │
  │  Output → DuckDB: raw_historical table                              │
  │         → data/raw.parquet  (file-system backup)                    │
  └───────────────────────────────┬─────────────────────────────────────┘
                                  │
                    ┌─────────────▼─────────────┐
                    │  QUALITY GATE #1           │
                    │  ✓ row_count  (FAIL→abort) │
                    │  ✓ freshness  (WARN)       │
                    └─────────────┬─────────────┘
                                  │
                                  ▼
  ┌─────────────────────────────────────────────────────────────────────┐
  │  STAGE 2 — CLEAN             (src/cleaning.py)                      │
  │                                                                     │
  │  handle_missing_values()  → ffill temp, zero precip, interpolate    │
  │  flag_outliers()          → IQR flags, rows kept                    │
  │  validate_date_continuity()→ gap audit per city                     │
  │                                                                     │
  │  Output → DuckDB: staging_historical table                          │
  │         → data/staging_historical.parquet                           │
  └───────────────────────────────┬─────────────────────────────────────┘
                                  │
                    ┌─────────────▼─────────────┐
                    │  QUALITY GATE #2           │
                    │  ✓ null_ratio    (WARN)    │
                    │  ✓ date_contin.  (WARN)    │
                    │  ✓ value_ranges  (WARN)    │
                    └─────────────┬─────────────┘
                                  │
                                  ▼
  ┌─────────────────────────────────────────────────────────────────────┐
  │  STAGE 3 — FEATURES          (src/features.py)                      │
  │                                                                     │
  │  add_rolling_averages()   → 7d / 30d means                          │
  │  add_seasonal_indicators()→ month, quarter, season                  │
  │  add_temperature_range()  → max − min                               │
  │  add_degree_days()        → HDD / CDD @ 18°C base                  │
  │  add_anomaly_score()      → deviation from DOY mean                 │
  │  add_lag_features()       → lag-1 / lag-2 temp & precip             │
  │                                                                     │
  │  Output → DuckDB: analytics_historical table                        │
  └───────────────────────────────┬─────────────────────────────────────┘
                                  │
                    ┌─────────────▼─────────────┐
                    │  QUALITY GATE #3           │
                    │  ✓ feature_completeness    │
                    │    (WARN if cols missing)  │
                    └─────────────┬─────────────┘
                                  │
                                  ▼
  ┌─────────────────────────────────────────────────────────────────────┐
  │  LOGGING  (logs/pipeline.log)                                       │
  │  • Start / end time & total duration                                │
  │  • Rows per stage per city                                          │
  │  • Errors & warnings                                                │
  │  • All quality gate results                                         │
  └─────────────────────────────────────────────────────────────────────┘

  Data layers in DuckDB
  ─────────────────────
  raw_historical       ← untouched API responses
  staging_historical   ← cleaned, nulls handled, outliers flagged
  analytics_historical ← ML-ready features added
```


## 7  CLI equivalents (for reference)

You can reproduce everything in this notebook from your terminal:

```bash
# Full historical re-ingest
python src/pipeline.py --mode full

# Incremental (only new days)
python src/pipeline.py --mode incremental

# Incremental for specific cities only
python src/pipeline.py --mode incremental --cities Baku Ganja

# Custom paths
python src/pipeline.py --mode full --db-path data/weather.duckdb --data-dir data --log-dir logs
```
